In [1]:
from __future__ import annotations

import gc
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor


# ============================================================
# CONFIGURATION
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

MODEL_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
)

EMBEDDING_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "embeddings"
    / "clip"
)

EMBEDDING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_NAME = "openai/clip-vit-base-patch32"
BATCH_SIZE = 16
MAX_TEXT_LENGTH = 77

ImageFile.LOAD_TRUNCATED_IMAGES = True


# ============================================================
# DEVICE
# ============================================================

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")

elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    DEVICE = torch.device("mps")

else:
    DEVICE = torch.device("cpu")

print("Using device:", DEVICE)
print("Model input root:", MODEL_INPUT_ROOT)
print("Embedding output root:", EMBEDDING_ROOT)


# ============================================================
# LOAD CLIP
# ============================================================

print("\nLoading CLIP model...")

processor = CLIPProcessor.from_pretrained(
    MODEL_NAME
)

model = CLIPModel.from_pretrained(
    MODEL_NAME
)

model = model.to(DEVICE)
model.eval()

print("CLIP model loaded successfully.")


# ============================================================
# HELPERS
# ============================================================

def load_image(
    image_path: str,
) -> Image.Image | None:

    try:
        with Image.open(image_path) as image:
            return image.convert("RGB")

    except Exception:
        return None


def normalize_embeddings(
    embeddings: torch.Tensor,
) -> torch.Tensor:

    denominator = embeddings.norm(
        p=2,
        dim=-1,
        keepdim=True,
    ).clamp(min=1e-12)

    return embeddings / denominator


def clear_device_cache() -> None:

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    elif DEVICE.type == "mps":
        torch.mps.empty_cache()


def generate_image_embeddings(
    pixel_values: torch.Tensor,
) -> torch.Tensor:

    vision_outputs = model.vision_model(
        pixel_values=pixel_values,
        return_dict=True,
    )

    image_features = model.visual_projection(
        vision_outputs.pooler_output
    )

    return normalize_embeddings(
        image_features
    )


def generate_text_embeddings(
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
) -> torch.Tensor:

    text_outputs = model.text_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        return_dict=True,
    )

    text_features = model.text_projection(
        text_outputs.pooler_output
    )

    return normalize_embeddings(
        text_features
    )


# ============================================================
# ENCODE ONE SPLIT
# ============================================================

def encode_split(
    split_name: str,
) -> None:

    print()
    print("=" * 80)
    print(f"ENCODING {split_name.upper()} DATASET")
    print("=" * 80)

    input_file = (
        MODEL_INPUT_ROOT
        / f"{split_name}.parquet"
    )

    if not input_file.exists():
        raise FileNotFoundError(
            f"Split file missing: {input_file}"
        )

    split_df = pd.read_parquet(
        input_file
    )

    required_columns = {
        "asin",
        "price",
        "absolute_path",
        "model_text",
        "dataset_split",
    }

    missing_columns = (
        required_columns
        - set(split_df.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Missing required columns: "
            f"{sorted(missing_columns)}"
        )

    print("Rows found:", f"{len(split_df):,}")

    image_embedding_batches: list[np.ndarray] = []
    text_embedding_batches: list[np.ndarray] = []

    metadata_rows: list[dict[str, Any]] = []
    failed_images: list[dict[str, Any]] = []

    for start_index in tqdm(
        range(
            0,
            len(split_df),
            BATCH_SIZE,
        ),
        desc=f"Encoding {split_name}",
    ):

        batch_df = split_df.iloc[
            start_index:
            start_index + BATCH_SIZE
        ]

        valid_images = []
        valid_texts = []
        valid_rows = []

        for row in batch_df.itertuples(
            index=False
        ):

            image_path = str(
                row.absolute_path
            ).strip()

            model_text = str(
                row.model_text
            ).strip()

            if not model_text:
                continue

            image = load_image(
                image_path
            )

            if image is None:
                failed_images.append(
                    {
                        "asin": row.asin,
                        "absolute_path": image_path,
                        "dataset_split": split_name,
                    }
                )
                continue

            valid_images.append(image)
            valid_texts.append(model_text)
            valid_rows.append(row)

        if not valid_rows:
            continue

        processed_inputs = processor(
            text=valid_texts,
            images=valid_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_TEXT_LENGTH,
        )

        pixel_values = processed_inputs[
            "pixel_values"
        ].to(DEVICE)

        input_ids = processed_inputs[
            "input_ids"
        ].to(DEVICE)

        attention_mask = processed_inputs[
            "attention_mask"
        ].to(DEVICE)

        with torch.inference_mode():

            image_features = (
                generate_image_embeddings(
                    pixel_values
                )
            )

            text_features = (
                generate_text_embeddings(
                    input_ids,
                    attention_mask,
                )
            )

        image_embedding_batches.append(
            image_features
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        text_embedding_batches.append(
            text_features
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        for row in valid_rows:
            metadata_rows.append(
                {
                    "asin": row.asin,
                    "price": row.price,
                    "absolute_path": row.absolute_path,
                    "model_text": row.model_text,
                    "dataset_split": row.dataset_split,
                }
            )

        del processed_inputs
        del pixel_values
        del input_ids
        del attention_mask
        del image_features
        del text_features

        clear_device_cache()

    if not image_embedding_batches:
        raise RuntimeError(
            f"No embeddings generated for {split_name}"
        )

    image_embedding_array = np.concatenate(
        image_embedding_batches,
        axis=0,
    )

    text_embedding_array = np.concatenate(
        text_embedding_batches,
        axis=0,
    )

    metadata_df = pd.DataFrame(
        metadata_rows
    )

    failed_images_df = pd.DataFrame(
        failed_images,
        columns=[
            "asin",
            "absolute_path",
            "dataset_split",
        ],
    )

    if not (
        len(image_embedding_array)
        == len(text_embedding_array)
        == len(metadata_df)
    ):
        raise RuntimeError(
            "Embedding and metadata row counts do not match."
        )

    np.save(
        EMBEDDING_ROOT
        / f"{split_name}_image_embeddings.npy",
        image_embedding_array,
    )

    np.save(
        EMBEDDING_ROOT
        / f"{split_name}_text_embeddings.npy",
        text_embedding_array,
    )

    metadata_df.to_parquet(
        EMBEDDING_ROOT
        / f"{split_name}_metadata.parquet",
        index=False,
        compression="snappy",
    )

    failed_images_df.to_csv(
        EMBEDDING_ROOT
        / f"{split_name}_failed_images.csv",
        index=False,
    )

    print(
        "Image embedding shape:",
        image_embedding_array.shape,
    )

    print(
        "Text embedding shape:",
        text_embedding_array.shape,
    )

    print(
        "Metadata rows:",
        len(metadata_df),
    )

    print(
        "Failed images:",
        len(failed_images_df),
    )

    del image_embedding_batches
    del text_embedding_batches
    del image_embedding_array
    del text_embedding_array
    del metadata_df
    del failed_images_df

    gc.collect()
    clear_device_cache()


# ============================================================
# RUN ALL SPLITS
# ============================================================

for split in [
    "train",
    "validation",
    "test",
]:
    encode_split(
        split_name=split
    )


print()
print("=" * 80)
print("CLIP EMBEDDING GENERATION COMPLETED")
print("=" * 80)

print(
    f"Embeddings saved under:\n"
    f"{EMBEDDING_ROOT}"
)

Using device: mps
Model input root: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/amazon_multimodal/model_input
Embedding output root: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/amazon_multimodal/embeddings/clip

Loading CLIP model...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP model loaded successfully.

ENCODING TRAIN DATASET
Rows found: 13,984


Encoding train:   0%|          | 0/874 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

...

Image embedding shape: (13984, 512)
Text embedding shape: (13984, 512)
Metadata rows: 13984
Failed images: 0

ENCODING VALIDATION DATASET
Rows found: 2,997


Encoding validation:   0%|          | 0/188 [00:00<?, ?it/s]

Image embedding shape: (2997, 512)
Text embedding shape: (2997, 512)
Metadata rows: 2997
Failed images: 0

ENCODING TEST DATASET
Rows found: 2,997


Encoding test:   0%|          | 0/188 [00:00<?, ?it/s]

Image embedding shape: (2997, 512)
Text embedding shape: (2997, 512)
Metadata rows: 2997
Failed images: 0

CLIP EMBEDDING GENERATION COMPLETED
Embeddings saved under:
/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/amazon_multimodal/embeddings/clip


In [2]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


# ============================================================
# CONFIGURATION
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

MODEL_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
)

EMBEDDING_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "embeddings"
    / "clip"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "price_prediction"
)

REPORT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "reports"
    / "price_prediction"
)

MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RESULTS_FILE = (
    REPORT_ROOT
    / "baseline_model_comparison.csv"
)

PREDICTIONS_FILE = (
    REPORT_ROOT
    / "test_predictions.csv"
)

BEST_MODEL_FILE = (
    MODEL_ROOT
    / "best_baseline_model.joblib"
)

CONFIG_FILE = (
    MODEL_ROOT
    / "baseline_model_config.json"
)

RANDOM_STATE = 42


# ============================================================
# HELPERS
# ============================================================

def load_split(split_name: str):
    image_embeddings = np.load(
        EMBEDDING_ROOT
        / f"{split_name}_image_embeddings.npy"
    )

    text_embeddings = np.load(
        EMBEDDING_ROOT
        / f"{split_name}_text_embeddings.npy"
    )

    metadata = pd.read_parquet(
        MODEL_INPUT_ROOT
        / f"{split_name}.parquet"
    ).reset_index(drop=True)

    if not (
        len(image_embeddings)
        == len(text_embeddings)
        == len(metadata)
    ):
        raise ValueError(
            f"Row mismatch for {split_name}: "
            f"image={len(image_embeddings)}, "
            f"text={len(text_embeddings)}, "
            f"metadata={len(metadata)}"
        )

    return (
        image_embeddings.astype(np.float32),
        text_embeddings.astype(np.float32),
        metadata,
    )


def evaluate_predictions(
    model_name: str,
    actual_price: np.ndarray,
    predicted_price: np.ndarray,
) -> dict[str, float]:

    return {
        "model": model_name,
        "mae": mean_absolute_error(
            actual_price,
            predicted_price,
        ),
        "rmse": mean_squared_error(
            actual_price,
            predicted_price,
        ) ** 0.5,
        "median_absolute_error": (
            median_absolute_error(
                actual_price,
                predicted_price,
            )
        ),
        "r2": r2_score(
            actual_price,
            predicted_price,
        ),
    }


# ============================================================
# LOAD DATA
# ============================================================

X_train_image, X_train_text, train_df = (
    load_split("train")
)

X_val_image, X_val_text, validation_df = (
    load_split("validation")
)

X_test_image, X_test_text, test_df = (
    load_split("test")
)

print("Train image:", X_train_image.shape)
print("Train text:", X_train_text.shape)
print("Train metadata:", train_df.shape)


# ============================================================
# TARGET
# ============================================================

y_train = (
    train_df["price"]
    .astype(float)
    .to_numpy()
)

y_validation = (
    validation_df["price"]
    .astype(float)
    .to_numpy()
)

y_test = (
    test_df["price"]
    .astype(float)
    .to_numpy()
)

y_train_log = np.log1p(y_train)


# ============================================================
# STRUCTURED FEATURES
# ============================================================

numeric_columns = [
    "stars",
    "reviews",
    "boughtInLastMonth",
    "isBestSeller",
    "cluster_id",
]

categorical_columns = [
    "category_name",
    "price_band",
]

for dataframe in [
    train_df,
    validation_df,
    test_df,
]:

    dataframe["reviews_log1p"] = np.log1p(
        pd.to_numeric(
            dataframe["reviews"],
            errors="coerce",
        ).fillna(0)
    )

    dataframe["bought_log1p"] = np.log1p(
        pd.to_numeric(
            dataframe[
                "boughtInLastMonth"
            ],
            errors="coerce",
        ).fillna(0)
    )

numeric_columns = [
    "stars",
    "reviews_log1p",
    "bought_log1p",
    "isBestSeller",
    "cluster_id",
]

structured_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_columns,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_columns,
        ),
    ],
    remainder="drop",
)

X_train_structured = (
    structured_preprocessor
    .fit_transform(train_df)
    .astype(np.float32)
)

X_validation_structured = (
    structured_preprocessor
    .transform(validation_df)
    .astype(np.float32)
)

X_test_structured = (
    structured_preprocessor
    .transform(test_df)
    .astype(np.float32)
)

joblib.dump(
    structured_preprocessor,
    MODEL_ROOT
    / "structured_preprocessor.joblib",
)

print(
    "Structured feature shape:",
    X_train_structured.shape,
)


# ============================================================
# CREATE EXPERIMENT FEATURE SETS
# ============================================================

feature_sets = {
    "structured_only": (
        X_train_structured,
        X_validation_structured,
        X_test_structured,
    ),
    "text_only": (
        X_train_text,
        X_val_text,
        X_test_text,
    ),
    "image_only": (
        X_train_image,
        X_val_image,
        X_test_image,
    ),
    "multimodal_fusion": (
        np.concatenate(
            [
                X_train_image,
                X_train_text,
                X_train_structured,
            ],
            axis=1,
        ),
        np.concatenate(
            [
                X_val_image,
                X_val_text,
                X_validation_structured,
            ],
            axis=1,
        ),
        np.concatenate(
            [
                X_test_image,
                X_test_text,
                X_test_structured,
            ],
            axis=1,
        ),
    ),
}


# ============================================================
# TRAIN BASELINE MODELS
# ============================================================

results = []
trained_models = {}
test_prediction_map = {}

for model_name, (
    X_train,
    X_validation,
    X_test,
) in feature_sets.items():

    print()
    print("=" * 80)
    print("TRAINING:", model_name)
    print("=" * 80)

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    model.fit(
        X_train,
        y_train_log,
    )

    validation_log_predictions = (
        model.predict(
            X_validation
        )
    )

    test_log_predictions = (
        model.predict(
            X_test
        )
    )

    validation_predictions = np.expm1(
        validation_log_predictions
    )

    test_predictions = np.expm1(
        test_log_predictions
    )

    validation_metrics = evaluate_predictions(
        f"{model_name}_validation",
        y_validation,
        validation_predictions,
    )

    test_metrics = evaluate_predictions(
        f"{model_name}_test",
        y_test,
        test_predictions,
    )

    results.extend(
        [
            validation_metrics,
            test_metrics,
        ]
    )

    trained_models[
        model_name
    ] = model

    test_prediction_map[
        model_name
    ] = test_predictions

    print(
        "Validation MAE:",
        round(
            validation_metrics["mae"],
            4,
        ),
    )

    print(
        "Validation RMSE:",
        round(
            validation_metrics["rmse"],
            4,
        ),
    )

    print(
        "Validation R²:",
        round(
            validation_metrics["r2"],
            4,
        ),
    )


# ============================================================
# SELECT BEST MODEL
# ============================================================

results_df = pd.DataFrame(
    results
)

results_df.to_csv(
    RESULTS_FILE,
    index=False,
)

validation_results = results_df[
    results_df["model"]
    .str.endswith("_validation")
].copy()

best_validation_row = (
    validation_results
    .sort_values(
        by="mae",
        ascending=True,
    )
    .iloc[0]
)

best_model_name = (
    best_validation_row["model"]
    .replace(
        "_validation",
        "",
    )
)

best_model = trained_models[
    best_model_name
]

joblib.dump(
    best_model,
    BEST_MODEL_FILE,
)

print()
print("Best model:", best_model_name)
print(
    "Best validation MAE:",
    best_validation_row["mae"],
)


# ============================================================
# SAVE TEST PREDICTIONS
# ============================================================

prediction_df = test_df[
    [
        "asin",
        "title",
        "category_name",
        "price",
        "price_band",
        "cluster_id",
    ]
].copy()

for model_name, predictions in (
    test_prediction_map.items()
):
    prediction_df[
        f"{model_name}_predicted_price"
    ] = predictions

    prediction_df[
        f"{model_name}_absolute_error"
    ] = np.abs(
        prediction_df["price"]
        - predictions
    )

prediction_df.to_csv(
    PREDICTIONS_FILE,
    index=False,
)


# ============================================================
# SAVE CONFIG
# ============================================================

model_config = {
    "target": "log1p(price)",
    "best_model": best_model_name,
    "random_state": RANDOM_STATE,
    "image_embedding_dimension": int(
        X_train_image.shape[1]
    ),
    "text_embedding_dimension": int(
        X_train_text.shape[1]
    ),
    "structured_feature_dimension": int(
        X_train_structured.shape[1]
    ),
    "experiments": list(
        feature_sets.keys()
    ),
}

with CONFIG_FILE.open(
    mode="w",
    encoding="utf-8",
) as file:
    json.dump(
        model_config,
        file,
        indent=2,
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("BASELINE MODEL TRAINING COMPLETED")
print("=" * 80)

display(
    results_df.sort_values(
        by=["model"]
    )
)

print("Best model saved:", BEST_MODEL_FILE)
print("Results saved:", RESULTS_FILE)
print("Predictions saved:", PREDICTIONS_FILE)

Train image: (13984, 512)
Train text: (13984, 512)
Train metadata: (13984, 26)
Structured feature shape: (13984, 251)

TRAINING: structured_only
Validation MAE: 11.2842
Validation RMSE: 77.7234
Validation R²: 0.5143

TRAINING: text_only
Validation MAE: 27.7244
Validation RMSE: 102.1825
Validation R²: 0.1605

TRAINING: image_only
Validation MAE: 29.6422
Validation RMSE: 105.5676
Validation R²: 0.104

TRAINING: multimodal_fusion
Validation MAE: 12.699
Validation RMSE: 82.0359
Validation R²: 0.4589

Best model: structured_only
Best validation MAE: 11.284235474333352

BASELINE MODEL TRAINING COMPLETED


,model,mae,rmse,median_absolute_error,r2
5,image_only_test,28.607050,87.947530,10.464303,0.139056
4,image_only_validation,29.642235,105.567649,10.608315,0.104005
7,multimodal_fusion_test,11.726647,57.423570,2.324959,0.632965
6,multimodal_fusion_validation,12.698999,82.035901,2.396131,0.458932
1,structured_only_test,10.043212,55.946729,2.009493,0.651601
0,structured_only_validation,11.284235,77.723405,2.070156,0.514323
3,text_only_test,27.317659,83.292891,9.510348,0.227776
2,text_only_validation,27.724437,102.182502,9.491885,0.160546


Best model saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/price_prediction/best_baseline_model.joblib
Results saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/reports/price_prediction/baseline_model_comparison.csv
Predictions saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/reports/price_prediction/test_predictions.csv
